In [ ]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from transformers import GLPNForDepthEstimation, GLPNImageProcessor
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

batch_size = 4
num_epochs = 50
learning_rate = 5e-5
patience = 7


In [ ]:
class DepthDataset(Dataset):
    def __init__(self, rgb_dir, depth_dir):
        self.rgb_files = sorted([
            os.path.join(rgb_dir,f) for f in os.listdir(rgb_dir)
            if f.endswith(('.png','.jpg','.jpeg'))
        ])
        self.depth_files = sorted([
            os.path.join(depth_dir,f) for f in os.listdir(depth_dir)
            if f.endswith(('.png','.jpg','.jpeg'))
        ])

        assert len(self.rgb_files) == len(self.depth_files)

    def __len__(self):
        return len(self.rgb_files)

    def __getitem__(self, idx):
        # --- RGB ---
        img = Image.open(self.rgb_files[idx]).convert("RGB")
        img = torch.from_numpy(np.array(img)).float().permute(2,0,1) / 255.0

        # --- Depth ---
        depth = Image.open(self.depth_files[idx])
        depth = np.array(depth).astype(np.float32)

        if depth.ndim == 3:
            depth = depth[:,:,0]

        depth[depth < 1] = 1.0     # éviter log(0)
        depth = np.log(depth)     # 🔥 CRUCIAL

        depth = torch.from_numpy(depth).unsqueeze(0)

        return img, depth


In [ ]:
rgb_dir   = "/content/drive/MyDrive/dataset1/RGB_aug"
depth_dir = "/content/drive/MyDrive/dataset1/Depth_aug"

dataset = DepthDataset(rgb_dir, depth_dir)

indices = list(range(len(dataset)))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

train_dataset = Subset(dataset, train_idx)
val_dataset   = Subset(dataset, val_idx)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)


In [ ]:
processor = GLPNImageProcessor.from_pretrained("vinvino02/glpn-nyu")
model = GLPNForDepthEstimation.from_pretrained("vinvino02/glpn-nyu").to(device)


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
criterion = nn.L1Loss()   # GLPN utilise L1


In [ ]:
best_val = float("inf")
counter = 0

for epoch in range(num_epochs):

    # -------- TRAIN --------
    model.train()
    train_loss = 0

    for imgs, depths in tqdm(train_loader, desc=f"Epoch {epoch+1} TRAIN"):
        imgs, depths = imgs.to(device), depths.to(device)

        imgs_pil = [
            Image.fromarray((img.permute(1,2,0).cpu().numpy()*255).astype(np.uint8))
            for img in imgs
        ]

        pixel_values = processor(images=imgs_pil, return_tensors="pt").pixel_values.to(device)

        optimizer.zero_grad()

        pred = model(pixel_values=pixel_values).predicted_depth
        if pred.dim()==3:
            pred = pred.unsqueeze(1)

        pred = F.interpolate(pred, size=depths.shape[-2:], mode="bilinear", align_corners=False)

        loss = criterion(pred, depths)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    # -------- VAL --------
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for imgs, depths in tqdm(val_loader, desc=f"Epoch {epoch+1} VAL"):
            imgs, depths = imgs.to(device), depths.to(device)

            imgs_pil = [
                Image.fromarray((img.permute(1,2,0).cpu().numpy()*255).astype(np.uint8))
                for img in imgs
            ]

            pixel_values = processor(images=imgs_pil, return_tensors="pt").pixel_values.to(device)

            pred = model(pixel_values=pixel_values).predicted_depth
            pred = F.interpolate(pred.unsqueeze(1), size=depths.shape[-2:], mode="bilinear", align_corners=False)

            val_loss += criterion(pred, depths).item()

    val_loss /= len(val_loader)

    print(f"\nEpoch {epoch+1}  Train: {train_loss:.4f}   Val: {val_loss:.4f}")

    # -------- Early stopping --------
    if val_loss < best_val:
        best_val = val_loss
        counter = 0
        torch.save(model.state_dict(), "/content/drive/MyDrive/best_glpn.pth")
        print("✅ Modèle sauvegardé")
    else:
        counter += 1
        print(f"⚠️ Pas d'amélioration ({counter}/{patience})")
        if counter >= patience:
            print("🛑 Early stopping")
            break


In [ ]:
import matplotlib.pyplot as plt

# 🔹 Tracer les courbes d'apprentissage
plt.figure(figsize=(10,6))
plt.plot(train_losses, label='Train Loss', marker='o')
plt.plot(val_losses, label='Validation Loss', marker='o')
plt.title("Courbes d'apprentissage du modèle GLPN")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()


test CNN


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PseudoDepthCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, 3, padding=1)

        self.conv4 = nn.Conv2d(64, 32, 3, padding=1)
        self.conv5 = nn.Conv2d(32, 16, 3, padding=1)
        self.out   = nn.Conv2d(16, 1, 3, padding=1)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        x = F.relu(self.conv4(x))
        x = F.relu(self.conv5(x))
        x = self.out(x)

        # Normalisation 0–1
        x = (x - x.min()) / (x.max() - x.min() + 1e-8)

        return x


In [ ]:
def add_vertical_bias(depth):
    B, C, H, W = depth.shape
    y = torch.linspace(0, 1, H).view(1, 1, H, 1).to(depth.device)
    return depth * 0.7 + y * 0.3


In [ ]:
from PIL import Image
import torchvision.transforms as T
import matplotlib.pyplot as plt

transform = T.Compose([
    T.Resize((256, 256)),
    T.ToTensor()
])

img = Image.open("/content/cap3.png").convert("RGB")
x = transform(img).unsqueeze(0)

model = PseudoDepthCNN()
model.eval()

with torch.no_grad():
    depth = model(x)
    depth = add_vertical_bias(depth)

depth_map = depth.squeeze().numpy()


In [ ]:
plt.imshow(depth_map, cmap="jet")
plt.colorbar()
plt.title("CNN baseline")
plt.show()


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import os

# Charger les images
img_rgb  = Image.open("/content/RBG1.png").convert("RGB")
img_glpn = Image.open("/content/cartePr1.png").convert("RGB")
img_cnn  = Image.open("/content/carteCNN1.png").convert("RGB")

# Redimensionner GLPN et CNN à la taille du RGB
W, H = img_rgb.size
img_glpn = img_glpn.resize((W, H), Image.BILINEAR)
img_cnn  = img_cnn.resize((W, H), Image.BILINEAR)

# Créer la figure
plt.figure(figsize=(15,5))

plt.subplot(1,3,1)
plt.imshow(img_rgb)
plt.title("Image RGB")
plt.axis("off")

plt.subplot(1,3,2)
plt.imshow(img_glpn)
plt.title("Carte profondeur GLPN")
plt.axis("off")

plt.subplot(1,3,3)
plt.imshow(img_cnn)
plt.title("Carte profondeur CNN")
plt.axis("off")

plt.tight_layout()

# Créer le dossier si nécessaire
os.makedirs("/content/sample_data", exist_ok=True)

# Sauvegarde
output_path = "/content/sample_data/comparaison_RGB_GLPN_CNN.png"
plt.savefig(output_path, dpi=300, bbox_inches="tight")

# Affichage
plt.show()

print(f"Image sauvegardée dans : {output_path}")
